# Classical time series analysis
This notebook tracks my efforts to find a competitive classical time series model that can serve as a benchmark against which I'd like to compare my quantum reservoir computing model.

In [1]:
# ---- load data ----
from src.data.loading.geosphere import Geosphere

data = Geosphere().load_data_into_memory()

historically_relevant_data = data[-1000:]

Loading data from file


In [25]:
# ---- select data ----

import pandas as pd

selected_columns = [
    'cglo'
    , 'chim'
    , 'dd'
    , 'ddx'
    , 'ff'
#    , 'ffam_flag'
    , 'ffx'
    , 'p'
#    , 'pred'
    , 'rf'
    , 'rr'
    , 'rrm'
    , 'sh'
    , 'so'
    , 'tb10'
    , 'tb20'
    , 'tb50'
    , 'tl'
    , 'tlmax'
    , 'tlmin'
    , 'ts'
    , 'tsmax'
    , 'tsmin'
    , 'zeitx'
    , 'timestamps'
#    , 'stationId'
]

usable_data = historically_relevant_data.dropna(axis="columns", how="all")

final_data_columns = usable_data.columns.intersection(selected_columns)

final_data = historically_relevant_data[final_data_columns].reset_index(drop=True)
final_data.set_index(pd.to_datetime(final_data['timestamps'], format="ISO8601"))
final_data.index.freq = '10min'

# output
print("Selected columns: {}".format(selected_columns))
print("Usable columns: {}".format(usable_data.columns))
print("Realised columns: {}".format(final_data_columns))

Selected columns: ['cglo', 'chim', 'dd', 'ddx', 'ff', 'ffx', 'p', 'rf', 'rr', 'rrm', 'sh', 'so', 'tb10', 'tb20', 'tb50', 'tl', 'tlmax', 'tlmin', 'ts', 'tsmax', 'tsmin', 'zeitx', 'timestamps']
Usable columns: Index(['cglo', 'dd', 'ddx', 'ff', 'ffam_flag', 'ffx', 'p', 'rf', 'rr', 'rrm',
       'so', 'tl', 'tlmax', 'tlmin', 'ts', 'tsmax', 'tsmin', 'zeitx',
       'timestamps', 'stationId'],
      dtype='object')
Realised columns: Index(['cglo', 'dd', 'ddx', 'ff', 'ffx', 'p', 'rf', 'rr', 'rrm', 'so', 'tl',
       'tlmax', 'tlmin', 'ts', 'tsmax', 'tsmin', 'zeitx', 'timestamps'],
      dtype='object')


In [28]:
# ---- model selection ----

# tried but failed due to conflicting dependencies:
# Seshadri, Ram (2020). GitHub - AutoViML/Auto_TS: enables you to build and deploy multiple time
# series models using ML and statistical techniques with a single line of code.
# Source code: https://github.com/AutoViML/Auto_TS

# instead used:
import pmdarima as pm

# selects best model and fits to data
classical_model = pm.auto_arima(final_data['tl'],
                      seasonal=True,
                      m=144, # 144 Intervalle à 10min = 24 Stunden Saison
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)

print(f"The type of the model is: {}".format(type(classical_model)))
print(f"Its parameters are: {}".format(classical_model))

In [35]:
# ---- model fitting ----
#MM: to be continued
forecast, conf_int = classical_model.predict(n_periods=50, return_conf_int=True)

print("Forecast: {}".format(forecast))
print("Conf int: {}".format(conf_int))


<class 'pmdarima.arima.arima.ARIMA'>
Forecast: 1000    1.965374
1001    1.933968
1002    1.905483
1003    1.879648
1004    1.856215
1005    1.834962
1006    1.815686
1007    1.798202
1008    1.782345
1009    1.767962
1010    1.754918
1011    1.743086
1012    1.732355
1013    1.722622
1014    1.713794
1015    1.705788
1016    1.698526
1017    1.691939
1018    1.685965
1019    1.680547
1020    1.675633
1021    1.671175
1022    1.667133
1023    1.663466
1024    1.660140
1025    1.657124
1026    1.654388
1027    1.651907
1028    1.649656
1029    1.647615
1030    1.645764
1031    1.644084
1032    1.642561
1033    1.641180
1034    1.639927
1035    1.638791
1036    1.637760
1037    1.636825
1038    1.635978
1039    1.635209
1040    1.634511
1041    1.633878
1042    1.633305
1043    1.632784
1044    1.632312
1045    1.631884
1046    1.631496
1047    1.631144
1048    1.630824
1049    1.630535
dtype: float64
Conf int: [[ 1.60673525  2.32401201]
 [ 1.40572941  2.46220623]
 [ 1.23414837  2.5768177